# CMDMamba GC — Rasterized Continuous Optuna

CMDMamba with rasterized short stream (VPIN heatmaps) + patched 15-min long stream.
USA session only (CT 08:30–15:45). Objective: **accuracy-weighted PnL**.

**Note**: SESSION_END=15:45 is the DST-safe US session boundary.
The raster file (`GC_rasterized.npz`) shifts between 07:00–15:45 (CDT/summer)
and 08:00–16:45 (CST/winter). Using 08:30–15:45 gives a consistent 30-bar
session grid with 100% raster coverage across all days.

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path and install (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow/
    !git pull
    %cd ..

    sys.path.insert(0, "/content/drive/MyDrive/CTAEnv/CTAFlow/")
    sys.path.insert(1, "/content/drive/MyDrive/CTAEnv/SierraPy")
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow is installed")

## 2. Imports

In [ ]:
import json, warnings, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.metrics import confusion_matrix, classification_report

from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df
from CTAFlow.models.prep.intraday_continuous import ContinuousIntradayPrep, SessionSpec
from CTAFlow.data.datasets.continuous import ContinuousRasterAlignedDataset, collate_continuous_raster
from CTAFlow.models.deep_learning.multi_branch.cmd_mamba import CMDMamba, CMDMambaConfig
from CTAFlow.models.deep_learning.training.loss import ProfitWeightedCE
from CTAFlow.models.deep_learning.training.backtest import backtest_eod_momentum

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}  |  torch {torch.__version__}  |  optuna {optuna.__version__}")

## 3. Configuration

In [ ]:
# --- Ticker / bar ---
TICKER = "GC"
BAR_RULE = "15min"
BAR_MIN = 15

# --- Session (USA, CT) ---
# 08:30-15:45 is the DST-safe window: raster covers 07:00-15:45 (CDT)
# and 08:00-16:45 (CST), so 08:30-15:45 gives 30 bars/day in both regimes.
SESSION_START = "08:30"
SESSION_END   = "15:45"

# --- Target ---
TARGET_HORIZON_MIN = 60
TARGET_STEPS = TARGET_HORIZON_MIN // BAR_MIN
NUM_CLASSES = 3
CLF_PERCENTILES = (30, 70)

# --- Sequence lengths (reduced 25% from original 256 / 32-128) ---
LONG_LOOKBACK = 192
SHORT_RANGE = (24, 96)
LONG_PATCH_OPTIONS = [2, 4, 8]

# --- Sampling stride ---
# stride=4 → sample every 4th bar (hourly) instead of every 15-min bar.
# Cuts training set ~75% with minimal info loss (adjacent bars share 93% of
# lookback and 75% of forward return). Validation stays dense (stride=1).
TRAIN_STRIDE = 4
VAL_STRIDE = 1

# --- Split / quality ---
VAL_SPLIT = 0.2
MAX_MISS_ROWS = 6
MAX_MISS_RATIO = 0.20

# --- Optuna ---
N_TRIALS = 30
TRIAL_EPOCHS = 20
FINAL_EPOCHS = 40
PATIENCE = 10

# --- Backtest costs (bps) ---
ENTRY_COST_BPS = 1.5
EXIT_COST_BPS = 1.0
ENTRY_SLIP_BPS = 1.0
EXIT_SLIP_BPS = 0.5

# --- Paths ---
if IN_COLAB:
    DATA_DIR = Path(f"/content/drive/MyDrive/features/{TICKER}")
else:
    DATA_DIR = Path(f"/workspace/model_data/{TICKER}")
DATA_PATH = DATA_DIR / "intraday.csv"
RASTER_PATH = DATA_DIR / f"{TICKER}_rasterized.npz"
RESULTS_PATH = DATA_DIR / "results" / "cmdmamba_gc_optuna"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# --- Download from S3 if not in Colab and data missing ---
if not IN_COLAB and (not DATA_PATH.exists() or not RASTER_PATH.exists()):
    import os
    from CTAFlow.data.storage.aws_client import AWSClient, S3Config

    _bucket = os.getenv("CTAFLOW_S3_BUCKET", "ctaflow-data")
    _region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    _endpoint = os.getenv("AWS_ENDPOINT_URL")

    print(f"Data missing locally - downloading {TICKER} from S3 ({_bucket})...")
    _s3 = AWSClient(S3Config(
        bucket_name=_bucket,
        region=_region,
        endpoint_url=_endpoint,
    ))
    _downloaded = _s3.download_ticker_data(
        ticker=TICKER,
        local_dir=str(DATA_DIR),
        data_types=["intraday", "rasterized"],
    )
    if not _downloaded:
        # Fall back to recursive directory download
        print(f"  Ticker data download returned empty, trying directory download...")
        _s3.download_directory(
            s3_prefix=f"workspace/model_data/{TICKER}/",
            local_dir=str(DATA_DIR),
        )
    print(f"  Downloaded: {list(_downloaded.keys()) if _downloaded else 'directory'}")

assert DATA_PATH.exists(), f"Missing {DATA_PATH}"
assert RASTER_PATH.exists(), f"Missing {RASTER_PATH}"

print(f"data:    {DATA_PATH}")
print(f"raster:  {RASTER_PATH}")
print(f"results: {RESULTS_PATH}")
print(f"stride:  train={TRAIN_STRIDE} (every {TRAIN_STRIDE*BAR_MIN}min)  val={VAL_STRIDE}")

## 4. Load and Prepare

Resample raw ticks to 15-min bars, build features via `ContinuousIntradayPrep` with
**in-module scaling** (`apply_scaling=True`), and classify the forward return into
3 classes (short / flat / long) using training-set percentiles.

In [ ]:
# ---- helpers: resample ----
def resample_bars(df, rule):
    agg = {"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"}
    for c in df.columns:
        if c not in agg and np.issubdtype(df[c].dtype, np.number):
            agg[c] = "last"
    return df.resample(rule).agg(agg).dropna(subset=["Open", "High", "Low", "Close"]).sort_index()


# ---- load raw data via read_exported_df and resample ----
raw = read_exported_df(str(DATA_PATH))
bars = resample_bars(raw, BAR_RULE)
print(f"bars: {len(bars)}  |  {bars.index.min()} -> {bars.index.max()}")

# ---- load raster ----
rz = np.load(RASTER_PATH, allow_pickle=False)
rdata, ridx = rz["data"], pd.to_datetime(rz["idx"])
RBINS = int(rdata.shape[2])
print(f"raster: {rdata.shape}  |  {ridx.min()} -> {ridx.max()}")

# ---- feature engineering (with in-module scaling) ----
# Two-session setup required: add_sessions() aliases session 1 -> is_london,
# session 2 -> is_usa. Using a single SessionSpec("USA",...) would overwrite
# is_usa=0. Keep both sessions so is_usa and sample_mode="usa" work correctly.
prep = ContinuousIntradayPrep(
    sessions=[
        SessionSpec("LONDON", "02:00", "11:00"),
        SessionSpec("USA", SESSION_START, SESSION_END),
    ],
    bar_minutes=BAR_MIN,
)
df, train_mask, tcols = prep.prepare(
    bars,
    steps_60m=TARGET_STEPS,
    keep_only_active=False,
    add_daily=True,
    add_overnight=True,
    add_deseas=True,
    add_time_features=True,
    add_resample_precalc=True,
    resample_rules=("30min", "60min"),
    rolling_days_deseas=252,
    refit_interval=10,
    use_legacy_deseas=False,
    apply_scaling=True,
    scale_to_basis_points=True,
)

# ---- filter to USA session only ----
active = df["is_usa"].astype(bool)
df = df[active].copy()
train_mask = train_mask.loc[df.index]

# ---- select feature columns ----
TARGET_COL = tcols[-1]
exclude = set(
    tcols
    + ["Open", "High", "Low", "Close", "Volume",
       "session_code", "is_active", "is_usa", "is_london"]
)
FEAT = [
    c for c in df.columns
    if c not in exclude
    and np.issubdtype(df[c].dtype, np.number)
    and pd.notna(df[c].std())
    and df[c].std() > 0
]

# ---- classify target ----
tr_returns = df.loc[train_mask, TARGET_COL].dropna()
q_low, q_high = np.percentile(tr_returns, CLF_PERCENTILES)

df["target_class"] = df[TARGET_COL].apply(
    lambda x: -1 if pd.isna(x) else (0 if x <= q_low else (1 if x <= q_high else 2))
).astype(np.int64)
df["target_return"] = df[TARGET_COL].astype(np.float32)
df = df[(df["target_class"] >= 0) & df["target_return"].notna()].copy()

print(f"target: {TARGET_COL}  |  q_low={q_low:.6f}  q_high={q_high:.6f}  |  features: {len(FEAT)}")
print((df["target_class"].value_counts(normalize=True).sort_index() * 100).round(2))

# ---- plot target distribution ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET_COL].dropna(), bins=80, alpha=0.75)
axes[0].axvline(q_low, c="r", ls="--", label=f"q_low={q_low:.4f}")
axes[0].axvline(q_high, c="g", ls="--", label=f"q_high={q_high:.4f}")
axes[0].legend()
axes[0].set_title("Target Distribution")

vc = df["target_class"].value_counts().sort_index()
axes[1].bar(["Short (0)", "Flat (1)", "Long (2)"], vc.values, color=["#e74c3c", "#95a5a6", "#27ae60"])
axes[1].set_title("Class Counts")
plt.tight_layout()
plt.show()

## 5. Alignment Diagnostic

Check why bar data and raster timestamps might not align before building the dataset.

In [ ]:
# --- Reproduce the exact normalisation the dataset constructor does ---
from CTAFlow.data.datasets.continuous import _normalize_ts_index, _build_session_index, _resample_frame

work = df.copy()
work.index = _normalize_ts_index(work.index)
work = work.sort_index()
work = _resample_frame(work, BAR_RULE)

rz = np.load(str(RASTER_PATH), allow_pickle=False)
raster_idx = _normalize_ts_index(pd.to_datetime(rz["idx"]))

print("=== Index timezone info ===")
print(f"  df.index.tz          : {df.index.tz}")
print(f"  work.index.tz        : {work.index.tz}")
print(f"  raster_idx.tz        : {raster_idx.tz}")

print(f"\n=== Date ranges ===")
print(f"  work   : {work.index.min()} -> {work.index.max()}  ({len(work)} rows)")
print(f"  raster : {raster_idx.min()} -> {raster_idx.max()}  ({len(raster_idx)} rows)")

work_days = set(work.index.normalize())
raster_days = set(raster_idx.normalize())
common_days = sorted(work_days & raster_days)
print(f"\n=== Day overlap ===")
print(f"  work days    : {len(work_days)}")
print(f"  raster days  : {len(raster_days)}")
print(f"  common days  : {len(common_days)}")

if not common_days:
    print("\n  NO COMMON DAYS \u2014 showing sample dates from each:")
    print(f"    work   (first 5): {sorted(work_days)[:5]}")
    print(f"    raster (first 5): {sorted(raster_days)[:5]}")
else:
    # Check a single common day end-to-end
    test_day = pd.Timestamp(common_days[len(common_days)//2])
    expected = _build_session_index(test_day, SESSION_START, SESSION_END, BAR_RULE)
    print(f"\n=== Sample day: {test_day.date()} ===")
    print(f"  expected timestamps ({len(expected)}): {expected[0]} -> {expected[-1]}")

    day_work = work.loc[work.index.normalize() == test_day]
    day_raster = raster_idx[raster_idx.normalize() == test_day]
    print(f"  work rows on this day : {len(day_work)}")
    print(f"  raster rows on this day: {len(day_raster)}")

    # How many expected timestamps exist in work / raster?
    work_hits = expected.isin(work.index).sum()
    raster_hits = expected.isin(raster_idx).sum()
    print(f"  expected in work   : {work_hits}/{len(expected)}")
    print(f"  expected in raster : {raster_hits}/{len(expected)}")

    if work_hits == 0 or raster_hits == 0:
        print("\n  Timestamps don't match expected session grid. Showing actual vs expected:")
        print(f"    expected[:5]      : {list(expected[:5])}")
        print(f"    work[:5] on day   : {list(day_work.index[:5])}")
        print(f"    raster[:5] on day : {list(day_raster[:5])}")

    # Run the full per-day loop to count drop reasons
    dropped_reasons = {}
    n_ok = 0
    for day in common_days:
        exp = _build_session_index(day, SESSION_START, SESSION_END, BAR_RULE)
        if len(exp) == 0:
            dropped_reasons[str(day.date())] = "empty_session_index"
            continue
        day_raw = work.reindex(exp)
        n_feat_miss = int(day_raw[FEAT].isna().all(axis=1).sum())
        ridx_pos = raster_idx.get_indexer(exp)
        n_raster_miss = int((ridx_pos < 0).sum())
        tlen = len(exp)
        miss_limit = max(MAX_MISS_ROWS, int(np.floor(MAX_MISS_RATIO * tlen)))

        if not np.any(ridx_pos >= 0):
            dropped_reasons[str(day.date())] = "no_raster_rows"
        elif n_feat_miss > miss_limit:
            dropped_reasons[str(day.date())] = f"feature_missing={n_feat_miss}/{tlen}"
        elif n_raster_miss > miss_limit:
            dropped_reasons[str(day.date())] = f"raster_missing={n_raster_miss}/{tlen}"
        else:
            n_ok += 1

    print(f"\n=== Per-day filter results ===")
    print(f"  passed : {n_ok}")
    print(f"  dropped: {len(dropped_reasons)}")
    if dropped_reasons:
        from collections import Counter
        reason_counts = Counter(v.split("=")[0] for v in dropped_reasons.values())
        print(f"  drop reasons: {dict(reason_counts)}")
        # show first few dropped
        for k, v in list(dropped_reasons.items())[:5]:
            print(f"    {k}: {v}")

## 6. Dataset and Loaders

Time-ordered train/val split at the day level. Training uses `sample_stride=4`
(hourly) to reduce autocorrelation and cut epoch time ~75%. Validation uses
dense stride=1 for granular evaluation. Training also randomises short-stream
length for augmentation; evaluation uses the fixed upper bound.

In [ ]:
ds_kwargs = dict(
    df=df,
    raster_npz_path=str(RASTER_PATH),
    feature_cols=FEAT,
    target_cols=["target_class", "target_return"],
    long_lookback=LONG_LOOKBACK,
    short_len_range=SHORT_RANGE,
    resample_rule=BAR_RULE,
    session_start=SESSION_START,
    session_end=SESSION_END,
    sample_mode="usa",
    allow_overlap=False,
    max_missing_rows_per_day=MAX_MISS_ROWS,
    max_missing_ratio_per_day=MAX_MISS_RATIO,
    return_meta=True,
)

# Training: stride=4 (hourly) for faster epochs + less autocorrelation
# Validation: stride=1 (dense) for granular evaluation and backtest
train_base = ContinuousRasterAlignedDataset(
    **ds_kwargs, random_short_len=True, sample_stride=TRAIN_STRIDE,
)
eval_base = ContinuousRasterAlignedDataset(
    **ds_kwargs, random_short_len=False, sample_stride=VAL_STRIDE,
)

# time-ordered split
sample_ts = pd.DatetimeIndex(eval_base.df.index[eval_base.sample_pos])
days = pd.Index(sample_ts.normalize().unique()).sort_values()
split_day = days[int(len(days) * (1 - VAL_SPLIT))]

tr_idx = np.flatnonzero(sample_ts < split_day)
va_idx = np.flatnonzero(sample_ts >= split_day)

# Training subset uses the strided train_base
train_ts = pd.DatetimeIndex(train_base.df.index[train_base.sample_pos])
tr_idx_strided = np.flatnonzero(train_ts < split_day)
train_ds = Subset(train_base, tr_idx_strided.tolist())
val_ds   = Subset(eval_base, va_idx.tolist())


def make_loaders(bs):
    pin = torch.cuda.is_available()
    tr = DataLoader(train_ds, batch_size=bs, shuffle=True,
                    num_workers=0, pin_memory=pin, collate_fn=collate_continuous_raster)
    va = DataLoader(val_ds, batch_size=bs, shuffle=False,
                    num_workers=0, pin_memory=pin, collate_fn=collate_continuous_raster)
    return tr, va


# quick sanity check
tr_loader, _ = make_loaders(16)
batch = next(iter(tr_loader))
xS, tS, xL, tL, mS, y, meta = batch

print(f"split:  {split_day.date()}  |  train: {len(train_ds)} (stride={TRAIN_STRIDE})  |  val: {len(val_ds)} (stride={VAL_STRIDE})")
print(f"time features: {eval_base.time_feature_cols}")
print(f"xS: {xS.shape}  xL: {xL.shape}  y: {y.shape}  raster_bins: {RBINS}")

## 7. Model, Training Helpers, and Optuna Objective

**Scoring**: `score = pnl * (accuracy / 100) * max(0, accuracy - baseline)`

Rewards profitable models proportionally to their accuracy and margin over baseline.
Accuracy weights are fixed (not tuned) so trial scores are directly comparable.

In [ ]:
TIME_DIM = len(eval_base.time_feature_cols)


# ---------------------------------------------------------------------------
# Model builder
# ---------------------------------------------------------------------------
def build_model(p):
    cfg = CMDMambaConfig(
        base_bar_minutes=BAR_MIN,
        long_patch=int(p["long_patch"]),
        long_stride=int(p["long_patch"]),
        n_long_layers=int(p["n_long_layers"]),
        n_short_layers=int(p["n_short_layers"]),
        raster_channels=4,
        raster_bins=RBINS,
        d_model=int(p["d_model"]),
        dropout=float(p["dropout"]),
        d_state_long=int(p["d_state_long"]),
        d_state_short=int(p["d_state_short"]),
        task="classification",
        out_dim=NUM_CLASSES,
    )
    return CMDMamba(long_input_dim=len(FEAT), time_feat_dim=TIME_DIM, cfg=cfg).to(device)


# ---------------------------------------------------------------------------
# Scoring: accuracy-weighted PnL
#   score = pnl * (accuracy / 100) * max(0, accuracy - baseline)
# accuracy and baseline are in percentage form (0-100).
# The margin term is in percentage points (effectively *100 vs fractions).
# ---------------------------------------------------------------------------
def compute_accuracy_weighted_pnl(accuracy, pnl, max_class_pct):
    acc_frac = accuracy / 100.0
    margin = max(0.0, accuracy - max_class_pct)
    return float(pnl * acc_frac * margin)


# ---------------------------------------------------------------------------
# Batch unpacking
# ---------------------------------------------------------------------------
def unpack(batch):
    if len(batch) == 7:
        xS, tS, xL, tL, mS, y, meta = batch
    else:
        xS, tS, xL, tL, mS, y = batch
        meta = None
    return (
        xS.to(device), tS.to(device), xL.to(device), tL.to(device),
        mS.to(device), y[:, 0].long().to(device), y[:, 1].float().to(device),
        meta,
    )


# ---------------------------------------------------------------------------
# Train / evaluate one epoch
# ---------------------------------------------------------------------------
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = total_correct = total_n = 0
    total_pnl = 0.0

    for batch in loader:
        xS, tS, xL, tL, mS, yc, yr, _ = unpack(batch)
        optimizer.zero_grad()
        out = model(x_short=xS, x_long=xL, t_short=tS, t_long=tL, short_mask=mS)

        loss = criterion(out, yc, returns=yr) if isinstance(criterion, ProfitWeightedCE) else criterion(out, yc)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        with torch.no_grad():
            probs = torch.softmax(out, dim=-1)
            preds = probs.argmax(dim=-1)
            position = probs[:, 2] - probs[:, 0]
            total_loss += loss.item() * yc.size(0)
            total_correct += (preds == yc).sum().item()
            total_pnl += (position * yr).sum().item()
            total_n += yc.size(0)

    n = max(1, total_n)
    return total_loss / n, 100.0 * total_correct / n, total_pnl / n


def evaluate(model, loader, criterion, ret_arrays=False):
    model.eval()
    total_loss = total_correct = total_n = 0
    total_pnl = 0.0
    all_logits, all_labels, all_returns, all_dates = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            xS, tS, xL, tL, mS, yc, yr, meta = unpack(batch)
            out = model(x_short=xS, x_long=xL, t_short=tS, t_long=tL, short_mask=mS)

            loss = criterion(out, yc, returns=yr) if isinstance(criterion, ProfitWeightedCE) else criterion(out, yc)
            probs = torch.softmax(out, dim=-1)
            preds = probs.argmax(dim=-1)
            position = probs[:, 2] - probs[:, 0]

            total_loss += loss.item() * yc.size(0)
            total_correct += (preds == yc).sum().item()
            total_pnl += (position * yr).sum().item()
            total_n += yc.size(0)

            if ret_arrays:
                all_logits.append(out.cpu().numpy())
                all_labels.append(yc.cpu().numpy())
                all_returns.append(yr.cpu().numpy())
                if meta is not None:
                    all_dates.extend(m.get("timestamp") if isinstance(m, dict) else None for m in meta)

    n = max(1, total_n)
    avg_loss = total_loss / n
    acc = 100.0 * total_correct / n
    avg_pnl = total_pnl / n

    if not ret_arrays:
        return avg_loss, acc, avg_pnl

    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    returns = np.concatenate(all_returns)
    dates = pd.to_datetime(np.array(all_dates)) if all_dates else None
    _, cnt = np.unique(labels, return_counts=True)
    baseline = 100.0 * cnt.max() / max(1, len(labels))

    return avg_loss, acc, avg_pnl, baseline, logits, labels, returns, dates


# ---------------------------------------------------------------------------
# Optuna objective
# ---------------------------------------------------------------------------
def objective(trial):
    p = {
        "batch_size":     trial.suggest_categorical("batch_size", [16, 32, 64]),
        "d_model":        trial.suggest_categorical("d_model", [64, 128, 256]),
        "n_long_layers":  trial.suggest_int("n_long_layers", 2, 4),
        "n_short_layers": trial.suggest_int("n_short_layers", 1, 3),
        "d_state_long":   trial.suggest_categorical("d_state_long", [32, 64, 128]),
        "d_state_short":  trial.suggest_categorical("d_state_short", [8, 16, 32]),
        "long_patch":     trial.suggest_categorical("long_patch", LONG_PATCH_OPTIONS),
        "dropout":        trial.suggest_float("dropout", 0.1, 0.4),
        "learning_rate":  trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True),
        "weight_decay":   trial.suggest_float("weight_decay", 0.01, 0.1, log=True),
        "profit_scale":   trial.suggest_float("profit_scale", 1.0, 20.0, log=True),
        "direction_penalty": trial.suggest_float("direction_penalty", 0.5, 2.0),
        "min_weight":     trial.suggest_float("min_weight", 0.5, 1.0),
        "max_weight":     trial.suggest_float("max_weight", 1.0, 5.0),
    }

    tr, va = make_loaders(int(p["batch_size"]))
    model = build_model(p)
    opt = optim.AdamW(model.parameters(), lr=p["learning_rate"], weight_decay=p["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TRIAL_EPOCHS)

    ce  = nn.CrossEntropyLoss()
    pce = ProfitWeightedCE(
        profit_scale=p["profit_scale"], min_weight=p["min_weight"],
        max_weight=p["max_weight"], direction_penalty=p["direction_penalty"],
    )

    best_score = -1e9
    patience_counter = 0
    best_metrics = {}

    for epoch in range(TRIAL_EPOCHS):
        crit = ce if epoch < 3 else pce
        train_epoch(model, tr, crit, opt)
        val_loss, val_acc, val_pnl, baseline, logits, labels, returns, dates = evaluate(
            model, va, crit, ret_arrays=True
        )
        scheduler.step()

        score = compute_accuracy_weighted_pnl(val_acc, val_pnl, baseline)
        acc_margin = val_acc - baseline

        if score > best_score:
            best_score = score
            patience_counter = 0

            bt_result = backtest_eod_momentum(
                predictions=logits, returns=returns, dates=dates,
                task="classification", long_class=2, short_class=0,
                entry_cost_bps=ENTRY_COST_BPS, exit_cost_bps=EXIT_COST_BPS,
                entry_slippage_bps=ENTRY_SLIP_BPS, exit_slippage_bps=EXIT_SLIP_BPS,
            )

            best_metrics = {
                "final_acc": val_acc,
                "final_loss": val_loss,
                "final_pnl": val_pnl,
                "max_class_pct": baseline,
                "acc_margin": acc_margin,
                "acc_weighted_score": score,
                "sharpe_like": bt_result.summary["sharpe_like"],
                "hit_rate": bt_result.summary["hit_rate"],
                "max_drawdown": bt_result.summary["max_drawdown"],
                "profit_factor": bt_result.trade_stats.get("profit_factor", 0.0),
                "n_trades": bt_result.trade_stats.get("n_trades", 0),
                "win_rate": bt_result.trade_stats.get("win_rate", 0.0),
            }
        else:
            patience_counter += 1

        print(f"  [CMDMamba] E{epoch+1:02} | Acc: {val_acc:.1f}% (base: {baseline:.1f}%, +{acc_margin:.1f}) | "
              f"PnL: {val_pnl:.6f} | Score: {score:.4f}")

        trial.report(score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= PATIENCE:
            break

    for k, v in best_metrics.items():
        trial.set_user_attr(k, float(v))
    trial.set_user_attr("n_params", sum(pp.numel() for pp in model.parameters()))

    return float(best_score)

## 8. Run Optimisation and Analyse Results

In [ ]:
STUDY_NAME = f"cmdmamba_{TICKER}_rasterized_clf"

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# --- Best trial summary ---
print("\n" + "=" * 60)
print("BEST TRIAL SUMMARY")
print("=" * 60)

best = study.best_trial
acc = best.user_attrs.get("final_acc", 0)
max_class = best.user_attrs.get("max_class_pct", 33.33)
acc_margin = best.user_attrs.get("acc_margin", acc - max_class)

print(f"\nAccuracy-Weighted PnL Score: {best.value:.4f}")
print(f"\n--- Accuracy Breakdown ---")
print(f"  Accuracy: {acc:.2f}%")
print(f"  Baseline (max class): {max_class:.2f}%")
print(f"  Margin over baseline: {acc_margin:+.2f}%")
print(f"\n--- Trading Metrics ---")
print(f"  Raw PnL: {best.user_attrs.get('final_pnl', 0):.6f}")
print(f"  Sharpe-like: {best.user_attrs.get('sharpe_like', 0):.3f}")
print(f"  Win Rate: {best.user_attrs.get('win_rate', 0):.2%}")
print(f"  Profit Factor: {best.user_attrs.get('profit_factor', 0):.2f}")
print(f"  # Trades: {best.user_attrs.get('n_trades', 0):.0f}")
print(f"  Max Drawdown: {best.user_attrs.get('max_drawdown', 0):.6f}")
print(f"\n--- Model Info ---")
print(f"  Parameters: {best.user_attrs.get('n_params', 0):,}")
print(f"\n--- Key Hyperparameters ---")
for k in ["d_model", "long_patch", "n_long_layers", "n_short_layers",
           "d_state_long", "d_state_short", "learning_rate", "weight_decay"]:
    if k in best.params:
        print(f"  {k}: {best.params[k]}")

# ---- trial analysis plots ----
tdf = study.trials_dataframe()
comp = tdf[tdf["state"] == "COMPLETE"].copy()
print(f"\nCompleted trials: {len(comp)}/{N_TRIALS}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(comp["value"], bins=20, alpha=0.7, edgecolor="black")
axes[0, 0].axvline(best.value, c="r", ls="--", label=f"best={best.value:.4f}")
axes[0, 0].legend()
axes[0, 0].set_title("Score Distribution")

axes[0, 1].plot(comp["number"], comp["value"], "o-", alpha=0.7)
axes[0, 1].set_title("Optimisation History")

for ax, col, title in [
    (axes[0, 2], "params_d_model",         "d_model vs Score"),
    (axes[1, 0], "params_long_patch",       "long_patch vs Score"),
    (axes[1, 1], "params_n_short_layers",   "short_layers vs Score"),
]:
    if col in comp.columns:
        ax.scatter(comp[col], comp["value"], alpha=0.7, s=70)
    ax.set_title(title)

if "params_learning_rate" in comp.columns:
    axes[1, 2].scatter(comp["params_learning_rate"], comp["value"], alpha=0.7, s=70)
    axes[1, 2].set_xscale("log")
axes[1, 2].set_title("Learning Rate vs Score")

for ax in axes.ravel():
    ax.grid(True, alpha=0.3)

plt.suptitle(f"CMDMamba Optuna \u2014 {TICKER} USA (Acc-Weighted PnL)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- parameter importance ----
imp = optuna.importance.get_param_importances(study)
idf = pd.DataFrame({"param": list(imp.keys()), "importance": list(imp.values())})

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=idf, x="importance", y="param", ax=ax)
ax.grid(True, alpha=0.3)
ax.set_title("Hyperparameter Importance")
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Final Training, Backtest, and Save

Retrain with best params using `CosineAnnealingWarmRestarts`, run cost-adjusted
backtest, save model checkpoint and artifacts.

In [ ]:
p = study.best_params.copy()
tr, va = make_loaders(int(p["batch_size"]))
model = build_model(p)
opt = optim.AdamW(model.parameters(), lr=p["learning_rate"], weight_decay=p["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
crit = ProfitWeightedCE(
    profit_scale=p["profit_scale"], min_weight=p["min_weight"],
    max_weight=p["max_weight"], direction_penalty=p["direction_penalty"],
)

print(f"Training final CMDMamba model...")
print(f"Parameters: {sum(pp.numel() for pp in model.parameters()):,}")
print(f"Weight decay: {p['weight_decay']:.4f}")
print(f"Train samples: {len(train_ds)} (stride={TRAIN_STRIDE})  |  Val samples: {len(val_ds)} (stride={VAL_STRIDE})")

hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_pnl": [], "val_score": []}
best_score = -1e9
best_state = None
patience_counter = 0

for epoch in range(FINAL_EPOCHS):
    tl, ta, tp = train_epoch(model, tr, crit, opt)
    vl, va_acc, vp, baseline, *_ = evaluate(model, va, crit, ret_arrays=True)
    sc = compute_accuracy_weighted_pnl(va_acc, vp, baseline)
    scheduler.step()

    hist["train_loss"].append(tl)
    hist["val_loss"].append(vl)
    hist["train_acc"].append(ta)
    hist["val_acc"].append(va_acc)
    hist["val_pnl"].append(vp)
    hist["val_score"].append(sc)

    if sc > best_score:
        best_score = sc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"E{epoch+1:02d}  loss={tl:.4f}/{vl:.4f}  acc={va_acc:.2f}% (base={baseline:.1f}%, +{va_acc-baseline:.1f})  pnl={vp:.6f}  score={sc:.4f}")
    if patience_counter >= PATIENCE:
        print(f"  -> early stop at epoch {epoch+1}")
        break

# ---- reload best checkpoint ----
if best_state is not None:
    model.load_state_dict(best_state)

# ---- final evaluation + backtest ----
vl, va_acc, vp, baseline, logits, ytrue, rets, dates = evaluate(model, va, crit, ret_arrays=True)

bt = backtest_eod_momentum(
    predictions=logits, returns=rets, dates=dates,
    task="classification", long_class=2, short_class=0,
    entry_cost_bps=ENTRY_COST_BPS, exit_cost_bps=EXIT_COST_BPS,
    entry_slippage_bps=ENTRY_SLIP_BPS, exit_slippage_bps=EXIT_SLIP_BPS,
)

print(f"\n{'=' * 60}")
print("FINAL BACKTEST RESULTS")
print(f"{'=' * 60}")
print(f"\nval loss={vl:.4f}  acc={va_acc:.2f}%  baseline={baseline:.2f}%  margin={va_acc - baseline:+.2f}%")
print(f"\nNet PnL: {bt.summary['net_pnl_sum']:.4f}")
print(f"Gross PnL: {bt.summary['gross_pnl_sum']:.4f}")
print(f"Total Costs: {bt.summary['cost_sum']:.4f}")
print(f"Sharpe-like: {bt.summary['sharpe_like']:.3f}")
print(f"Max Drawdown: {bt.summary['max_drawdown']:.4f}")
print(f"\nTrades: {bt.trade_stats['n_trades']:.0f}")
print(f"Win Rate: {bt.trade_stats['win_rate']:.2%}")
print(f"Profit Factor: {bt.trade_stats['profit_factor']:.2f}")

# ---- training curves ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(hist["train_loss"], label="train")
axes[0, 0].plot(hist["val_loss"], ls="--", label="val")
axes[0, 0].legend(); axes[0, 0].set_title("Loss")

axes[0, 1].plot(hist["train_acc"], label="train")
axes[0, 1].plot(hist["val_acc"], ls="--", label="val")
axes[0, 1].legend(); axes[0, 1].set_title("Accuracy (%)")

axes[1, 0].plot(hist["val_pnl"], label="val_pnl", c="green")
axes[1, 0].plot(hist["val_score"], label="acc_wt_pnl score", c="purple")
axes[1, 0].legend(); axes[1, 0].set_title("PnL / Score")

bt.frame["cum_pnl"].plot(ax=axes[1, 1], c="black")
axes[1, 1].set_title("Backtest Cumulative PnL")

for ax in axes.ravel():
    ax.grid(True, alpha=0.3)
plt.suptitle(f"CMDMamba Final \u2014 {TICKER} USA", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_training_backtest.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- confusion matrix ----
pred_labels = logits.argmax(axis=1)
cm = confusion_matrix(ytrue, pred_labels, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Short", "Flat", "Long"], yticklabels=["Short", "Flat", "Long"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(ytrue, pred_labels, target_names=["Short", "Flat", "Long"], digits=4))

# ---- save all artifacts ----
joblib.dump(study, RESULTS_PATH / f"{STUDY_NAME}_study.pkl")
tdf.to_csv(RESULTS_PATH / f"{STUDY_NAME}_trials.csv", index=False)

best_params_out = {
    **study.best_params,
    "best_value": float(study.best_value),
    "ticker": TICKER,
    "model": "CMDMamba",
    "target_col": TARGET_COL,
    "target_steps": int(TARGET_STEPS),
    "q_low": float(q_low),
    "q_high": float(q_high),
    "feature_cols": FEAT,
    "time_feature_cols": list(eval_base.time_feature_cols),
    "long_lookback": int(LONG_LOOKBACK),
    "short_len_range": list(SHORT_RANGE),
    "raster_bins": int(RBINS),
    "bar_rule": BAR_RULE,
    "session": f"USA CT {SESSION_START}-{SESSION_END}",
    "session_start": SESSION_START,
    "session_end": SESSION_END,
    "train_stride": TRAIN_STRIDE,
    "val_stride": VAL_STRIDE,
    "scoring": "pnl * (acc/100) * max(0, acc - baseline)",
    "backtest_summary": bt.summary,
    "backtest_trade_stats": bt.trade_stats,
}
with open(RESULTS_PATH / f"{STUDY_NAME}_best_params.json", "w") as f:
    json.dump(best_params_out, f, indent=2, default=float)

torch.save({
    "model_state_dict": model.state_dict(),
    "params": study.best_params,
    "history": hist,
    "feature_cols": FEAT,
    "time_feature_cols": list(eval_base.time_feature_cols),
    "clf_thresholds": {"q_low": float(q_low), "q_high": float(q_high)},
}, RESULTS_PATH / f"{STUDY_NAME}_final_model.pth")

bt.frame.to_csv(RESULTS_PATH / f"{STUDY_NAME}_backtest_frame.csv")

print(f"\nsaved to {RESULTS_PATH}")
for pth in sorted(RESULTS_PATH.glob(f"{STUDY_NAME}*")):
    print(f"  - {pth.name}")